# Fase 3 — Notebook 1: Split Holdout 80/20 por Paciente

**Objetivo:** Dividir los 79 pacientes en train (80%) y test (20%) de forma
estratificada, respetando el ratio de clases y agrupando TODOS los segmentos
de un mismo paciente en el mismo split.

**No se mueven archivos.** El resultado es un JSON por filtro+duración con
las listas de stems de train y test. Los notebooks siguientes filtran las
rutas originales usando esos stems.

**Salida:**
```
Ansiedad/fase3/splits/
    split_{filtro}_{dur}s.json
    split_summary.txt
```

Cada JSON tiene la forma:
```json
{
  "filter": "log-mel",
  "duration": "3s",
  "random_state": 42,
  "test_size": 0.2,
  "train": {
    "stems": [...],
    "labels": [...],
    "n_class0": 48,
    "n_class1": 15
  },
  "test": {
    "stems": [...],
    "labels": [...],
    "n_class0": 12,
    "n_class1": 4
  }
}
```

---
**IMPORTANTE:** Este notebook solo se corre UNA VEZ.
Si el JSON ya existe, lo lee y muestra el resumen sin regenerarlo,
garantizando que el split sea siempre el mismo.

In [10]:
# ─── Celda 1: Imports y configuración ─────────────────────────────────────
import re
import json
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit

SPECS_ROOT   = Path("Clasificación Final/Ansiedad/specs")
SPLITS_DIR   = Path("Ansiedad/fase3/splits")
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

FILTER_CONFIGS = [
    {"name": "log-mel",   "img_h": 128},
    {"name": "cqt",       "img_h": 168},
    {"name": "pcen",      "img_h": 128},
    {"name": "gammatone", "img_h": 128},
]
DURATIONS    = [2, 3, 4, 10]
RANDOM_STATE = 42       # semilla fija — NUNCA cambiar
TEST_SIZE    = 0.20     # 20% para test

print(f"Raiz de espectrogramas : {SPECS_ROOT}")
print(f"Directorio de splits   : {SPLITS_DIR}")
print(f"Random state           : {RANDOM_STATE}")
print(f"Test size              : {TEST_SIZE*100:.0f}%")
print()

Raiz de espectrogramas : Clasificación Final/Ansiedad/specs
Directorio de splits   : Ansiedad/fase3/splits
Random state           : 42
Test size              : 20%



In [11]:
# ─── Celda 2: Función para construir el índice de stems ───────────────────

def build_file_index(root: Path):
    """
    Lee las carpetas 0/ y 1/ dentro de root y devuelve:
      stems  : lista ordenada de IDs de paciente (sin sufijo _segXX)
      labels : array de etiquetas correspondientes
      img_map: dict stem -> [Path, Path, ...] (todos sus segmentos)
    """
    img_map    = {}
    stem_label = {}
    for cls_str in ["0", "1"]:
        d = root / cls_str
        if not d.exists():
            print(f"  [AVISO] No existe: {d}")
            continue
        for p in sorted(d.glob("*.png")):
            stem = re.sub(r'_seg\d+$', '', p.stem)
            img_map.setdefault(stem, []).append(p)
            stem_label[stem] = int(cls_str)
    stems  = sorted(img_map.keys())
    labels = np.array([stem_label[s] for s in stems], dtype=np.int64)
    return stems, labels, img_map

print("Funcion build_file_index definida")

Funcion build_file_index definida


In [12]:
# ─── Celda 3: Generar o cargar splits ─────────────────────────────────────
#
# Para cada combinacion filtro x duración:
#   - Si el JSON ya existe → lo carga (split inmutable)
#   - Si no existe → hace el split y lo guarda
#
# El split se hace UNA SOLA VEZ sobre los stems (nivel paciente),
# con StratifiedShuffleSplit para respetar el ratio de clases.

all_splits = {}   # { "log-mel_3s": {"train":{...}, "test":{...}}, ... }

for filt_cfg in FILTER_CONFIGS:
    filter_name = filt_cfg["name"]

    for dur in DURATIONS:
        key      = f"{filter_name}_{dur}s"
        json_path = SPLITS_DIR / f"split_{key}.json"

        # ── Si el JSON ya existe, solo cargar ─────────────────────────────
        if json_path.exists():
            with open(json_path) as f:
                split_data = json.load(f)
            all_splits[key] = split_data
            print(f"[CARGADO]  {key:25s}  "
                  f"train={split_data['train']['n_class0']+split_data['train']['n_class1']} pac "
                  f"(0:{split_data['train']['n_class0']} / 1:{split_data['train']['n_class1']})  "
                  f"test={split_data['test']['n_class0']+split_data['test']['n_class1']} pac "
                  f"(0:{split_data['test']['n_class0']} / 1:{split_data['test']['n_class1']})")
            continue

        # ── Si no existe, construir el índice y hacer el split ────────────
        image_root = SPECS_ROOT / filter_name / f"{dur}s"
        if not image_root.exists():
            print(f"[SKIP]     {key:25s}  No existe: {image_root}")
            continue

        stems, labels, img_map = build_file_index(image_root)
        if len(stems) == 0:
            print(f"[SKIP]     {key:25s}  Sin imágenes")
            continue

        # StratifiedShuffleSplit a nivel de paciente
        sss = StratifiedShuffleSplit(
            n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
        train_idx, test_idx = next(sss.split(stems, labels))

        train_stems  = [stems[i]  for i in train_idx]
        train_labels = labels[train_idx].tolist()
        test_stems   = [stems[i]  for i in test_idx]
        test_labels  = labels[test_idx].tolist()

        split_data = {
            "filter"      : filter_name,
            "duration"    : f"{dur}s",
            "random_state": RANDOM_STATE,
            "test_size"   : TEST_SIZE,
            "specs_root"  : str(SPECS_ROOT),
            "train": {
                "stems"   : train_stems,
                "labels"  : train_labels,
                "n_class0": int(sum(1 for l in train_labels if l == 0)),
                "n_class1": int(sum(1 for l in train_labels if l == 1)),
            },
            "test": {
                "stems"   : test_stems,
                "labels"  : test_labels,
                "n_class0": int(sum(1 for l in test_labels if l == 0)),
                "n_class1": int(sum(1 for l in test_labels if l == 1)),
            }
        }

        with open(json_path, "w") as f:
            json.dump(split_data, f, indent=2)

        all_splits[key] = split_data
        print(f"[GENERADO] {key:25s}  "
              f"train={len(train_stems)} pac "
              f"(0:{split_data['train']['n_class0']} / 1:{split_data['train']['n_class1']})  "
              f"test={len(test_stems)} pac "
              f"(0:{split_data['test']['n_class0']} / 1:{split_data['test']['n_class1']})")

print("\nSplits completados")

[GENERADO] log-mel_2s                 train=63 pac (0:48 / 1:15)  test=16 pac (0:12 / 1:4)
[GENERADO] log-mel_3s                 train=63 pac (0:48 / 1:15)  test=16 pac (0:12 / 1:4)
[GENERADO] log-mel_4s                 train=63 pac (0:48 / 1:15)  test=16 pac (0:12 / 1:4)
[GENERADO] log-mel_10s                train=63 pac (0:48 / 1:15)  test=16 pac (0:12 / 1:4)
[GENERADO] cqt_2s                     train=63 pac (0:48 / 1:15)  test=16 pac (0:12 / 1:4)
[GENERADO] cqt_3s                     train=63 pac (0:48 / 1:15)  test=16 pac (0:12 / 1:4)
[GENERADO] cqt_4s                     train=63 pac (0:48 / 1:15)  test=16 pac (0:12 / 1:4)
[GENERADO] cqt_10s                    train=63 pac (0:48 / 1:15)  test=16 pac (0:12 / 1:4)
[GENERADO] pcen_2s                    train=63 pac (0:48 / 1:15)  test=16 pac (0:12 / 1:4)
[GENERADO] pcen_3s                    train=63 pac (0:48 / 1:15)  test=16 pac (0:12 / 1:4)
[GENERADO] pcen_4s                    train=63 pac (0:48 / 1:15)  test=16 pac (0:12 / 1:4)

In [13]:
# ─── Celda 4: Resumen global y verificaciones ──────────────────────────────

print("="*72)
print("RESUMEN DE SPLITS — FASE 3")
print("="*72)
print()

# Verificar que todos los splits del mismo filtro tienen los mismos pacientes
# (deberían, ya que el RANDOM_STATE es fijo)
for filt_cfg in FILTER_CONFIGS:
    filter_name = filt_cfg["name"]
    print(f"Filtro: {filter_name}")
    print(f"  {'Dur':5s}  {'Train pac':10s}  {'Train C0':9s}  {'Train C1':9s}  "
          f"{'Test pac':9s}  {'Test C0':8s}  {'Test C1':8s}  {'Ratio train':12s}")
    print(f"  {'─'*75}")

    for dur in DURATIONS:
        key = f"{filter_name}_{dur}s"
        if key not in all_splits:
            print(f"  {dur}s    [sin datos]")
            continue

        sp = all_splits[key]
        tr = sp["train"]; te = sp["test"]
        tr_total = tr["n_class0"] + tr["n_class1"]
        te_total = te["n_class0"] + te["n_class1"]
        ratio = tr["n_class0"] / max(tr["n_class1"], 1)
        print(f"  {dur}s     {tr_total:5d}        {tr['n_class0']:5d}       "
              f"{tr['n_class1']:5d}       {te_total:5d}      "
              f"{te['n_class0']:4d}      {te['n_class1']:4d}      "
              f"{ratio:.2f}:1")
    print()

# Verificar que no hay fuga entre train y test (ningún stem aparece en los dos)
print("Verificacion de no-fuga (data leakage):")
leakage_found = False
for key, sp in all_splits.items():
    train_set = set(sp["train"]["stems"])
    test_set  = set(sp["test"]["stems"])
    overlap   = train_set & test_set
    if overlap:
        print(f"  [ERROR] {key}: {len(overlap)} stems en ambos splits! -> {overlap}")
        leakage_found = True

if not leakage_found:
    print("  OK — ningun paciente aparece en train y test a la vez")

# Verificar consistencia entre duraciones del mismo filtro
print()
print("Verificacion de consistencia entre duraciones:")
for filt_cfg in FILTER_CONFIGS:
    filter_name = filt_cfg["name"]
    keys_filt   = [f"{filter_name}_{d}s" for d in DURATIONS if f"{filter_name}_{d}s" in all_splits]
    if len(keys_filt) < 2:
        continue
    ref_train = set(all_splits[keys_filt[0]]["train"]["stems"])
    ref_test  = set(all_splits[keys_filt[0]]["test"]["stems"])
    consistent = True
    for k in keys_filt[1:]:
        if set(all_splits[k]["train"]["stems"]) != ref_train:
            print(f"  [AVISO] {filter_name}: train stems difieren entre duraciones")
            consistent = False
            break
    if consistent:
        print(f"  OK — {filter_name}: mismo split en todas las duraciones")

print()
print("Archivos JSON guardados en:")
for f in sorted(SPLITS_DIR.glob("*.json")):
    print(f"  {f}")

RESUMEN DE SPLITS — FASE 3

Filtro: log-mel
  Dur    Train pac   Train C0   Train C1   Test pac   Test C0   Test C1   Ratio train 
  ───────────────────────────────────────────────────────────────────────────
  2s        63           48          15          16        12         4      3.20:1
  3s        63           48          15          16        12         4      3.20:1
  4s        63           48          15          16        12         4      3.20:1
  10s        63           48          15          16        12         4      3.20:1

Filtro: cqt
  Dur    Train pac   Train C0   Train C1   Test pac   Test C0   Test C1   Ratio train 
  ───────────────────────────────────────────────────────────────────────────
  2s        63           48          15          16        12         4      3.20:1
  3s        63           48          15          16        12         4      3.20:1
  4s        63           48          15          16        12         4      3.20:1
  10s        63         

In [14]:
# ─── Celda 5: Función helper para los notebooks siguientes ────────────────
#
# Esta función se puede copiar directamente a los otros notebooks de Fase 3
# para cargar el split de forma estandarizada.

def load_split(filter_name: str, duration: int,
               splits_dir: str = "Ansiedad/fase3/splits"):
    """
    Carga el split pre-generado para un filtro y duración dados.

    Retorna:
        train_stems  : list[str]
        train_labels : np.ndarray int64
        test_stems   : list[str]
        test_labels  : np.ndarray int64
    """
    json_path = Path(splits_dir) / f"split_{filter_name}_{duration}s.json"
    if not json_path.exists():
        raise FileNotFoundError(
            f"Split no encontrado: {json_path}\n"
            f"Corre primero fase3_split_holdout.ipynb")

    with open(json_path) as f:
        sp = json.load(f)

    train_stems  = sp["train"]["stems"]
    train_labels = np.array(sp["train"]["labels"], dtype=np.int64)
    test_stems   = sp["test"]["stems"]
    test_labels  = np.array(sp["test"]["labels"], dtype=np.int64)

    print(f"Split cargado: {filter_name} {duration}s")
    print(f"  Train: {len(train_stems)} pacientes "
          f"(0:{sp['train']['n_class0']} / 1:{sp['train']['n_class1']})")
    print(f"  Test : {len(test_stems)} pacientes "
          f"(0:{sp['test']['n_class0']} / 1:{sp['test']['n_class1']})")

    return train_stems, train_labels, test_stems, test_labels


# Test rapido de la funcion
print("Prueba de la funcion load_split:")
print()
try:
    tr_s, tr_l, te_s, te_l = load_split("log-mel", 2)
    print(f"  Primer stem train : {tr_s[0]}")
    print(f"  Primer stem test  : {te_s[0]}")
    print()
    print("OK — funcion lista para copiar a los otros notebooks")
except FileNotFoundError as e:
    print(f"  [AVISO] {e}")

Prueba de la funcion load_split:

Split cargado: log-mel 2s
  Train: 63 pacientes (0:48 / 1:15)
  Test : 16 pacientes (0:12 / 1:4)
  Primer stem train : 032_recortado_denoised
  Primer stem test  : 070_recortado_denoised

OK — funcion lista para copiar a los otros notebooks


In [15]:
# ─── Celda 6: Resumen de segmentos por split ──────────────────────────────
# Muestra cuantos PNGs (segmentos) hay en train y test para cada config,
# y cuantos sintéticos habrá que generar para llegar a ratio 1:1 en train.

import re

def count_segments(stems, img_map):
    return sum(len(img_map[s]) for s in stems if s in img_map)

def build_file_index_simple(root: Path):
    img_map    = {}
    stem_label = {}
    for cls_str in ["0", "1"]:
        d = root / cls_str
        if not d.exists(): continue
        for p in sorted(d.glob("*.png")):
            stem = re.sub(r'_seg\d+$', '', p.stem)
            img_map.setdefault(stem, []).append(p)
            stem_label[stem] = int(cls_str)
    return img_map

print("="*80)
print("SEGMENTOS POR SPLIT Y SINTÉTICOS NECESARIOS PARA RATIO 1:1 EN TRAIN")
print("="*80)
print(f"{'Config':25s}  {'Train C0 segs':14s}  {'Train C1 segs':14s}  "
      f"{'Sinteticos':12s}  {'Test C0':8s}  {'Test C1':8s}")
print("─"*80)

for filt_cfg in FILTER_CONFIGS:
    filter_name = filt_cfg["name"]
    for dur in DURATIONS:
        key = f"{filter_name}_{dur}s"
        if key not in all_splits: continue

        sp         = all_splits[key]
        image_root = SPECS_ROOT / filter_name / f"{dur}s"
        if not image_root.exists(): continue

        img_map = build_file_index_simple(image_root)

        tr_stems_c0 = [s for s, l in zip(sp["train"]["stems"], sp["train"]["labels"]) if l == 0]
        tr_stems_c1 = [s for s, l in zip(sp["train"]["stems"], sp["train"]["labels"]) if l == 1]
        te_stems_c0 = [s for s, l in zip(sp["test"]["stems"],  sp["test"]["labels"])  if l == 0]
        te_stems_c1 = [s for s, l in zip(sp["test"]["stems"],  sp["test"]["labels"])  if l == 1]

        segs_tr_c0 = count_segments(tr_stems_c0, img_map)
        segs_tr_c1 = count_segments(tr_stems_c1, img_map)
        segs_te_c0 = count_segments(te_stems_c0, img_map)
        segs_te_c1 = count_segments(te_stems_c1, img_map)

        # Sinteticos necesarios para igualar C0 en train
        sinteticos = max(0, segs_tr_c0 - segs_tr_c1)

        print(f"{key:25s}  {segs_tr_c0:8d}       {segs_tr_c1:8d}       "
              f"{sinteticos:8d}      {segs_te_c0:5d}   {segs_te_c1:5d}")

print()
print("Nota: 'Sinteticos' = segmentos que el VAE debera generar para")
print("      que clase 1 train = clase 0 train (ratio 1:1)")
print()
print("FASE 3 NOTEBOOK 1 COMPLETADO")
print(f"JSONs en: {SPLITS_DIR}")

SEGMENTOS POR SPLIT Y SINTÉTICOS NECESARIOS PARA RATIO 1:1 EN TRAIN
Config                     Train C0 segs   Train C1 segs   Sinteticos    Test C0   Test C1 
────────────────────────────────────────────────────────────────────────────────
log-mel_2s                     7665           2783           4882       1927     491
log-mel_3s                     5090           1847           3243       1281     327
log-mel_4s                     3802           1381           2421        957     243
log-mel_10s                     751            272            479        189      47
cqt_2s                         7665           2783           4882       1927     491
cqt_3s                         5090           1847           3243       1281     327
cqt_4s                         3802           1381           2421        957     243
cqt_10s                         751            272            479        189      47
pcen_2s                        7665           2783           4882       1927   